# MLP

In [ ]:
import numpy as np
import pandas as pd
from plt_rcs import *
import torch, os

In [ ]:
plt.rc(group='figure', figsize=(4,4))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

In [ ]:
os.getcwd()

In [ ]:
os.chdir('../../data')

In [ ]:
[i for i in os.listdir() if i.endswith('pkl') and 'MNIST' in i][0]

In [ ]:
objs = pd.read_pickle('MNIST_Dataset.pkl')

In [ ]:
globals().update(objs)

In [ ]:
%whos

In [ ]:
test_mnist, train_mnist = test_mnist, train_mnist

## 시드 고정 함수

In [ ]:
import random

In [ ]:
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
set_seed()

## 배치 데이터 생성

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
bs = 256

In [ ]:
train_loader = DataLoader(
    dataset=train_mnist,
    batch_size=bs,
    shuffle=True
)

In [ ]:
test_loader = DataLoader(
    dataset=test_mnist,
    batch_size=bs
)

## 미니 배치 데이터 형태 확인

In [ ]:
batch_images, batch_labels = next(iter(train_loader))

In [ ]:
batch_images.shape

In [ ]:
flattenws_images = batch_images.reshape(batch_images.shape[0], -1)

In [ ]:
flattenws_images.shape

## 다층 퍼셉트론 모델 생성

In [ ]:
import torch.nn as nn

In [ ]:
model = nn.Sequential(
    nn.Linear(in_features=784, out_features=256),
    nn.Sigmoid(),
    nn.Linear(in_features=256, out_features=10)
).to(device)

In [ ]:
model[0].weight.shape

## 손실 함수와 최적화 알고리즘 생성

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

## 미니 배치 준비 함수 생성

In [ ]:
def prepare_batch(images, labels, flatten=True):
    if flatten:
        images = images.reshape(images.shape[0], -1)
    x = images.to(device) # 입력 특성은 순전파 학습을 위해 학습 장치로 이동
    y = labels.to(device) # 출력 벡터는 손실 함수 계산을 위해 학습 장치로 이동
    return x, y

## 미니 배치 학습 함수 생성

In [ ]:
def train_batch(model, x, y):
    logits = model(x)
    loss = criterion(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

## 미니 배치 정확도 평가 함수 생성

In [ ]:
@torch.no_grad()
def accuracy_batch(model, data_loader):
    model.eval()
    n_sample, correct = 0, 0
    for images, labels in data_loader:
        x, y = prepare_batch(images, labels)
        batch_size = y.shape[0]
        n_sample += batch_size # 미니 배치 크기를 n_sample에 누적
        logits = model(x)
        y_pred = logits.argmax(dim=1) # 미니 배치 정확도 계산
        correct += (y == y_pred).sum().item() # 실제값과 예측값이 일치하는 건수를 correct에 누적
    acc = correct / n_sample
    return acc

## 다층 퍼셉트론 모델 학습

In [ ]:
train_losses = []
train_accs = []
test_accs = []

epochs = 10

for epoch in range(epochs):
    model.train # 모델 학습 모드로 전환
    n_sample, loss_sum = 0, 0.0
    # 미니 배치 = 1 Epoch
    for images, labels in train_loader:
        x, y = prepare_batch(images, labels)
        batch_size = y.shape[0]
        n_sample += batch_size
        batch_loss = train_batch(model, x, y)
        loss_sum += batch_loss * batch_size # 배치 크기가 매번 다르므로 가중합
        
    train_loss = loss_sum / n_sample # 에포크 손실값 평균 계산
    train_losses.append(train_loss)

    train_acc = accuracy_batch(model, train_loader)
    train_accs.append(train_acc)

    test_acc = accuracy_batch(model, test_loader)
    test_accs.append(test_acc)

    print(f'[Epoch {epoch+1:02d}] Loss = {train_loss:.4f}, ', f'Train_Acc = {train_acc:.4f}, Test_Acc = {test_acc:.4f}')

## Trainer 클래스 임포트

In [ ]:
os.getcwd()

In [ ]:
os.chdir('../code/dl')

In [ ]:
[i for i in os.listdir() if '.py' in i]

In [ ]:
from trainer import Trainer, set_seed

In [ ]:
set_seed(0)

In [ ]:
model = nn.Sequential(
    nn.Linear(in_features=784, out_features=256),
    nn.Sigmoid(),
    nn.Linear(in_features=256, out_features=10)
)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

## Trainer 클래스 사용

In [ ]:
trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=test_loader
)

In [ ]:
histoty = trainer.fit()

## 학습 결과 시각화

In [ ]:
trainer.plot_loss()

In [ ]:
trainer.plot_accuracy()

In [ ]:
trainer.plot_confusion_matrix(data_loader=test_loader)

In [ ]:
trainer.plot_misclassified(data_loader=test_loader)

## ReLU 활성화 함수

In [ ]:
model = nn.Sequential(
    nn.Linear(in_features=784, out_features=256),
    nn.ReLU(),
    nn.Linear(in_features=256, out_features=10),
).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [ ]:
trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=test_loader
)
histoty = trainer.fit()

## 결과 시각화

In [ ]:
trainer.plot_loss()

In [ ]:
trainer.plot_accuracy()

In [ ]:
trainer.plot_confusion_matrix(data_loader=test_loader)

In [ ]:
trainer.plot_misclassified(data_loader=test_loader)

## 드롭아웃 추가

In [ ]:
model = nn.Sequential(
    nn.Linear(in_features=784, out_features=256),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(in_features=256, out_features=10),
).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [ ]:
trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=test_loader
)

In [ ]:
history = trainer.fit(n_epochs=10)

## 학습 결과 시각화

In [ ]:
trainer.plot_loss()

In [ ]:
trainer.plot_accuracy()

In [ ]:
trainer.plot_confusion_matrix(data_loader=test_loader)

In [ ]:
trainer.plot_misclassified(data_loader=test_loader)

## 은닉층 추가

In [ ]:
model = nn.Sequential(
    nn.Linear(in_features=784, out_features=256),
    nn.ReLU(),
    nn.Linear(in_features=256, out_features=128),
    nn.ReLU(),
    nn.Linear(in_features=128, out_features=10),
).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [ ]:
trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=test_loader
)
history = trainer.fit(n_epochs=10)

## 학습 결과 시각화

In [ ]:
trainer.plot_loss()

In [ ]:
trainer.plot_accuracy()

In [ ]:
trainer.plot_confusion_matrix(data_loader=test_loader)

In [ ]:
trainer.plot_misclassified(data_loader=test_loader)